# How to route between sub-chains

:::info Prerequisites

This guide assumes familiarity with the following concepts:
- [LangChain Expression Language (LCEL)](/docs/concepts/lcel)
- [Chaining runnables](/docs/how_to/sequence/)
- [Configuring chain parameters at runtime](/docs/how_to/configure)
- [Prompt templates](/docs/concepts/prompt_templates)
- [Chat Messages](/docs/concepts/messages)

:::

Routing allows you to create non-deterministic chains where the output of a previous step defines the next step. Routing can help provide structure and consistency around interactions with models by allowing you to define states and use information related to those states as context to model calls.

There are two ways to perform routing:

1. Conditionally return runnables from a [`RunnableLambda`](/docs/how_to/functions) (recommended)
2. Using a `RunnableBranch` (legacy)

We'll illustrate both methods using a two step sequence where the first step classifies an input question as being about `LangChain`, `Anthropic`, or `Other`, then routes to a corresponding prompt chain.

## Example Setup
First, let's create a chain that will identify incoming questions as being about `LangChain`, `Anthropic`, or `Other`:

In [13]:
from langchain_ollama import OllamaLLM 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

chain = (
    PromptTemplate.from_template(
        """Given the user question below, classify it as either being about `LangChain`, `Anthropic`, or `Other`.

Do not respond with more than one word.

<question>
{question}
</question>

Classification:"""
    )
    # | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")
# ollama run llama3.2:3b-instruct-q4_K_M
    | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")
    | StrOutputParser()
)

chain.invoke({"question": "how do I call Anthropic?"})

'Anthropic'

Now, let's create three sub chains:

In [14]:
langchain_chain = PromptTemplate.from_template(
    """You are an expert in langchain. \
Always answer questions starting with "As Harrison Chase told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")
anthropic_chain = PromptTemplate.from_template(
    """You are an expert in anthropic. \
Always answer questions starting with "As Dario Amodei told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")
general_chain = PromptTemplate.from_template(
    """Respond to the following question:

Question: {question}
Answer:"""
) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")

## Using a custom function (Recommended)

You can also use a custom function to route between different outputs. Here's an example:

In [15]:
def route(info):
    if "anthropic" in info["topic"].lower():
        return anthropic_chain
    elif "langchain" in info["topic"].lower():
        return langchain_chain
    else:
        return general_chain

In [16]:
from langchain_core.runnables import RunnableLambda

full_chain = {"topic": chain, "question": lambda x: x["question"]} | RunnableLambda(
    route
)

In [17]:
full_chain.invoke({"question": "how do I use Anthropic?"})

'As Dario Amodei told me, using Anthropic is relatively straightforward, but it does require some technical expertise and familiarity with machine learning concepts. To get started, you\'ll need to create an account on the Anthropic website or access their platform through a supported framework like TensorFlow or PyTorch.\n\nFirst, you\'ll need to install the necessary dependencies and frameworks required for Anthropic\'s models. Once installed, you can import the Anthropic library into your Python script and begin using their pre-trained models.\n\nOne of the most common use cases is natural language processing (NLP), where you can leverage Anthropic\'s transformer-based models like RoBERTa or BART to perform tasks such as text classification, sentiment analysis, and machine translation. To do this, you\'ll typically need to preprocess your input data, define a custom model configuration, and then pass in your input data to the Anthropic API.\n\nFor example, to use Anthropic\'s RoBERT

In [18]:
full_chain.invoke({"question": "how do I use LangChain?"})

'As Harrison Chase told me, using LangChain involves several steps that enable you to integrate it into your project seamlessly.\n\nFirstly, you need to install LangChain by running `pip install langchain` in your terminal. This will download the necessary dependencies and packages for LangChain.\n\nNext, create a new Python file (e.g., `langchain_example.py`) and import the required modules: `from langchain import LLMChain`, `from langchain.chains.base import BaseLLMChain`, and `from langchain.utils import print_chain`.\n\nThen, define your chain by creating an instance of the `BaseLLMChain` class. This is where you can specify the type of chain you want to create (e.g., `base_llm_chain`). You can customize this chain further by passing in additional parameters.\n\nAfter that, use the `run_in_process()` function to run your LLM pipeline in a separate process. This allows for better performance and scalability.\n\nFor example:\n```python\nfrom langchain import LLMChain\nfrom langchain.

In [19]:
full_chain.invoke({"question": "whats 2 + 2"})

'4.'

## Using a RunnableBranch

A `RunnableBranch` is a special type of runnable that allows you to define a set of conditions and runnables to execute based on the input. It does **not** offer anything that you can't achieve in a custom function as described above, so we recommend using a custom function instead.

A `RunnableBranch` is initialized with a list of (condition, runnable) pairs and a default runnable. It selects which branch by passing each condition the input it's invoked with. It selects the first condition to evaluate to True, and runs the corresponding runnable to that condition with the input.

If no provided conditions match, it runs the default runnable.

Here's an example of what it looks like in action:

In [20]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "anthropic" in x["topic"].lower(), anthropic_chain),
    (lambda x: "langchain" in x["topic"].lower(), langchain_chain),
    general_chain,
)
full_chain = {"topic": chain, "question": lambda x: x["question"]} | branch
full_chain.invoke({"question": "how do I use Anthropic?"})

'As Dario Amodei told me, using Anthropic requires a deep understanding of its core principles and functionality. To get started, you\'ll need to familiarize yourself with the platform\'s interface and explore its various features.\n\nFirstly, create an account on Anthropic\'s website and navigate to the dashboard. From there, you can access a range of tools and resources designed to help you build and deploy AI models.\n\nOne of the primary functions of Anthropic is its Model Hub, where you can browse and interact with pre-trained AI models. To use the Model Hub, simply click on "Models" in the navigation bar, select a model, and explore its capabilities.\n\nTo build your own AI model from scratch, use Anthropic\'s Model Builder tool. This interface allows you to configure various parameters and experiment with different architectures before deploying your model.\n\nAdditionally, Anthropic provides access to its Dataset Marketplace, where you can find and download large-scale datasets

In [21]:
full_chain.invoke({"question": "how do I use LangChain?"})

"As Harrison Chase told me, using LangChain typically involves several key steps. First, you'll need to install the necessary packages, including LangChain itself and any other dependencies required by your project.\n\nOnce installed, you can create a new LangChain model by instantiating the `langchain_model` class. This will allow you to define the structure of your language model, including the type of prompts, responses, and any additional data sources.\n\nTo fine-tune or train your model, you'll use the `train` method provided by LangChain. This allows you to pass in your dataset and configure various training parameters, such as learning rate and batch size.\n\nAfter training, you can use the `predict` method to generate responses based on a given prompt. This is typically done by calling the `prompt_to_response` function on an instance of the trained model.\n\nLangChain also provides a variety of utilities for integrating with other tools and platforms, such as support for multip

In [25]:
full_chain.invoke({"question": "whats 2 + 2"})

'4.'

## Routing by semantic similarity

One especially useful technique is to use embeddings to route a query to the most relevant prompt. Here's an example.

In [26]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import OpenAIEmbeddings

physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)


def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)


chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")
    | StrOutputParser()
)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
print(chain.invoke("What's a black hole"))

Using PHYSICS
As a physics professor, I would be happy to provide a concise and easy-to-understand explanation of what a black hole is.

A black hole is an incredibly dense region of space-time where the gravitational pull is so strong that nothing, not even light, can escape from it. This means that if you were to get too close to a black hole, you would be pulled in and crushed by the intense gravitational forces.

The formation of a black hole occurs when a massive star, much larger than our Sun, reaches the end of its life and collapses in on itself. This collapse causes the matter to become extremely dense, and the gravitational force becomes so strong that it creates a point of no return, known as the event horizon.

Beyond the event horizon, the laws of physics as we know them break down, and the intense gravitational forces create a singularity, which is a point of infinite density and curvature in space-time.

Black holes are fascinating and mysterious objects, and there is st

In [ ]:
print(chain.invoke("What's a path integral"))

Using MATH
A path integral is a powerful mathematical concept in physics, particularly in the field of quantum mechanics. It was developed by the renowned physicist Richard Feynman as an alternative formulation of quantum mechanics.

In a path integral, instead of considering a single, definite path that a particle might take from one point to another, as in classical mechanics, the particle is considered to take all possible paths simultaneously. Each path is assigned a complex-valued weight, and the total probability amplitude for the particle to go from one point to another is calculated by summing (integrating) over all possible paths.

The key ideas behind the path integral formulation are:

1. Superposition principle: In quantum mechanics, particles can exist in a superposition of multiple states or paths simultaneously.

2. Probability amplitude: The probability amplitude for a particle to go from one point to another is calculated by summing the complex-valued weights of all po

## Next steps

You've now learned how to add routing to your composed LCEL chains.

Next, check out the other how-to guides on runnables in this section.